# Extract info from PDFs — v2

follows four stages: **ingest → preprocess → pattern-match → validate**, and adds explicit source traceability (`source_file`, `page_number`) and a validation layer, as recommended there.

Target document: `Sample2025 Credit Card Statements.pdf` — a synthetic U.S. Bank-style statement:
* Two dates per transaction (`Post Date`, `Trans Date`), each `MM/DD` **without a year**.
* A reference number column.
* A single amount per line (no cashback column).
* Transactions split across two sections — *Purchases and Other Debits* and *Payments and Other Credits* — which is the only place the sign of each amount is implied.
* Transactions continue across two separate pages.

## Import libraries

In [1]:
import re
from pathlib import Path
from datetime import date, datetime

import pypdf
import pandas as pd

## View PDF text
### View actual PDF

In [2]:
PDF_PATH = Path('Sample2025 Credit Card Statements.pdf')


# Function to print the content of a PDF document
def print_pdf_content(pdf_path):
    with open(pdf_path, 'rb') as file:
        reader = pypdf.PdfReader(file)
        for page_num in range(len(reader.pages)):
            print(f"Page {page_num + 1}:\n")
            page = reader.pages[page_num]
            page_text = page.extract_text()
            print(page_text)
            print("\n" + "#" * 100 + "\n")  # Print a separator between pages


# Now call the function to print the content of the document
print_pdf_content(PDF_PATH)

Page 1:

1010101010101010101010101111110000001011101000110000100110101001111101000000110110010100001001011010100011101100001010011000100000110010010010010111110110111011110011000011111001001100101000100100110110000000100100100000101010000100000010111000001100111011010010100000110000001011100011011100000010001001100000110011100110111010101101001100110001011110111101011000100110111011001011111111111111111111
Open Date: 12/17/2024 Closing Date: 01/15/2025 Account: **** **** **** 8897
Page 1 of 3
1-866-485-4545
CRESC CITY HARBOR DST (CPN 001643647)
Earned This Statement $23.24
Reward Dollars Available $4,767.07
For details, see your rewards summary.
Previous Balance + $4,248.00
Payments - $4,248.00
Other Credits $0.00
Purchases + $2,324.22
Balance Transfers $0.00
Advances $0.00
Other Debits $0.00
Fees Charged $0.00
Interest Charged $0.00
Credit Line $14,000.00
Available Credit $11,675.78
Days in Billing Period 30
.
.
New Balance $2,324.22
Minimum Payment Due $1,163.00
Payment Due Date 02/1

## Functions definition
### Document ingestion

In [3]:
def extract_pdf_pages(pdf_path: Path) -> list[dict]:
    """Extract page-level text while retaining source metadata for traceability."""
    reader = pypdf.PdfReader(str(pdf_path))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        pages.append(
            {
                "source_file": pdf_path.name,
                "page_number": page_number,
                "raw_text": page.extract_text() or "",
            }
        )
    return pages

### Statement metadata

Transaction lines only carry `MM/DD`, so the statement's `Open Date` / `Closing Date` and account number are pulled from page 1 first.

In [4]:
PERIOD_PATTERN = re.compile(
    r"Open Date:\s*(?P<open>\d{2}/\d{2}/\d{4})\s*"
    r"Closing Date:\s*(?P<close>\d{2}/\d{2}/\d{4})",
)

ACCOUNT_PATTERN = re.compile(r"Account:\s*\*{4}\s*\*{4}\s*\*{4}\s*(?P<last4>\d{4})")


def extract_statement_metadata(pages: list[dict]) -> dict:
    """Pull statement period and account number from page 1 for year resolution and traceability."""
    first_page_text = pages[0]["raw_text"]

    period_match = PERIOD_PATTERN.search(first_page_text)
    account_match = ACCOUNT_PATTERN.search(first_page_text)

    period_start = (
        datetime.strptime(period_match.group("open"), "%m/%d/%Y").date()
        if period_match else None
    )
    period_end = (
        datetime.strptime(period_match.group("close"), "%m/%d/%Y").date()
        if period_match else None
    )

    return {
        "period_start": period_start,
        "period_end": period_end,
        "account_last4": account_match.group("last4") if account_match else None,
    }


def resolve_year(month_day: str, period_start: date, period_end: date) -> int:
    """A transaction only carries MM/DD. Pick whichever candidate year keeps
    the date inside the statement period (statements can cross a year end)."""
    month, day = (int(part) for part in month_day.split("/"))
    for year in {period_start.year, period_end.year}:
        candidate = date(year, month, day)
        if period_start <= candidate <= period_end:
            return year
    return period_end.year

### Text normalization

In [5]:
def normalize_text(value: str) -> str:
    """Standardize PDF text before applying transaction patterns."""
    value = value.replace("\u00a0", " ")
    value = value.replace("\r", "\n")
    value = re.sub(r"[ \t]+", " ", value)
    return value.strip()

### Transaction pattern

Example lines the pattern needs to match:
```
12/19 12/18 5389 USPS PO 0518780457    CRESCENT CITY CA $73.00
01/08 01/08 ET PAYMENT   THANK YOU $4,248.00
```
Two `MM/DD` dates, a reference code (digits or letters like `ET`), a free-text description, and a single dollar amount that may include a thousands comma. `SECTION_MARKERS` is used to track which of the two transaction sections (*Purchases* vs *Payments*) a line falls under, since that determines the sign.

In [6]:
TRANSACTION_PATTERN = re.compile(
    r"(?P<post_date>\d{2}/\d{2})\s+"
    r"(?P<trans_date>\d{2}/\d{2})\s+"
    r"(?P<ref>[A-Za-z0-9]+)\s+"
    r"(?P<description>.*?)\s+"
    r"\$(?P<amount>[\d,]+\.\d{2})",
    flags=re.IGNORECASE,
)

SECTION_MARKERS = {
    "Purchases and Other Debits": "Purchase",
    "Payments and Other Credits": "Payment/Credit",
}

### Parsing and traceability

In [7]:
def parse_transactions(pages: list[dict], metadata: dict) -> pd.DataFrame:
    records = []
    current_section = None

    for page in pages:
        for line in normalize_text(page["raw_text"]).split("\n"):
            for marker, label in SECTION_MARKERS.items():
                if marker in line:
                    current_section = label

            match = TRANSACTION_PATTERN.search(line)
            if not match:
                continue

            data = match.groupdict()
            year = resolve_year(
                data["post_date"], metadata["period_start"], metadata["period_end"]
            )
            post_date = pd.to_datetime(
                f"{data['post_date']}/{year}", format="%m/%d/%Y", errors="coerce"
            )

            amount = float(data["amount"].replace(",", ""))
            section = current_section or "Unclassified"
            signed_amount = -amount if section == "Payment/Credit" else amount

            records.append(
                {
                    "source_file": page["source_file"],
                    "page_number": page["page_number"],
                    "account_last4": metadata["account_last4"],
                    "post_date": post_date,
                    "trans_date": data["trans_date"],
                    "ref": data["ref"],
                    "description": data["description"].strip(),
                    "amount": signed_amount,
                    "type": section,
                    "matched_text": match.group(0).strip(),
                }
            )

    return pd.DataFrame.from_records(records)

### Validation layer

In [8]:
def validate_transactions(df: pd.DataFrame) -> pd.DataFrame:
    validated = df.copy()

    validated["is_complete"] = (
        validated[["post_date", "description", "amount"]].notna().all(axis=1)
    )
    validated["is_duplicate"] = validated.duplicated(
        subset=["source_file", "post_date", "ref", "description", "amount"],
        keep=False,
    )
    validated["requires_review"] = (
        ~validated["is_complete"] | validated["is_duplicate"]
    )

    return validated

## Running the pipeline against `Sample2025 Credit Card Statements.pdf`

In [9]:
pages = extract_pdf_pages(PDF_PATH)
metadata = extract_statement_metadata(pages)
metadata

{'period_start': datetime.date(2024, 12, 17), 'period_end': datetime.date(2025, 1, 15), 'account_last4': '8897'}

In [10]:
transactions_df = parse_transactions(pages, metadata)
transactions_df

                              source_file  ...                                       matched_text
0   Sample2025 Credit Card Statements.pdf  ...  12/19 12/18 5389 USPS PO 0518780457 CRESCENT C...
1   Sample2025 Credit Card Statements.pdf  ...  12/23 12/19 1157 ELK VALLEY FUEL MART CRESCENT...
2   Sample2025 Credit Card Statements.pdf  ...  12/23 12/20 3538 CANVA* I04372-0513440 CAMDEN ...
3   Sample2025 Credit Card Statements.pdf  ...  12/27 12/26 1664 ADOBE *ADOBE 4085366000 CA $1...
4   Sample2025 Credit Card Statements.pdf  ...  12/30 12/28 7434 Amazon.com*ZE55N46D0 Amzn.com...
5   Sample2025 Credit Card Statements.pdf  ...   12/30 12/29 4396 DOCKWA.COM NEWPORT RI $1,062.50
6   Sample2025 Credit Card Statements.pdf  ...  12/31 12/30 4688 USPS PO 0518780457 CRESCENT C...
7   Sample2025 Credit Card Statements.pdf  ...  01/02 12/30 7924 ELK VALLEY FUEL MART CRESCENT...
8   Sample2025 Credit Card Statements.pdf  ...  01/06 01/03 3208 TMOBILE*AUTO PAY 800-937-8997...
9   Sample2025 Credi

In [11]:
validated_df = validate_transactions(transactions_df)
validated_df['requires_review'].value_counts()

requires_review
False    15
Name: count, dtype: int64

## Reconciliation against statement control totals

Totals printed on the statement itself (`Purchases + $2,324.22`, `Payments - $4,248.00`) are extracted independently and compared against the sums of the parsed transactions.

In [ ]:
PURCHASES_PATTERN = re.compile(r"Purchases\s*\+\s*\$(?P<purchases>[\d,]+\.\d{2})")
PAYMENTS_PATTERN = re.compile(r"Payments\s*-\s*\$(?P<payments>[\d,]+\.\d{2})")

first_page_text = pages[0]["raw_text"]
control_purchases = float(PURCHASES_PATTERN.search(first_page_text).group("purchases").replace(",", ""))
control_payments = float(PAYMENTS_PATTERN.search(first_page_text).group("payments").replace(",", ""))

extracted_purchases = transactions_df.loc[transactions_df["type"] == "Purchase", "amount"].sum()
extracted_payments = -transactions_df.loc[transactions_df["type"] == "Payment/Credit", "amount"].sum()

reconciliation = pd.DataFrame(
    {
        "control_total": [control_purchases, control_payments],
        "extracted_total": [extracted_purchases, extracted_payments],
    },
    index=["Purchases", "Payments"],
)
reconciliation["difference"] = (reconciliation["control_total"] - reconciliation["extracted_total"]).round(2)
reconciliation

           control_total  extracted_total  difference
Purchases        2324.22          2324.22        -0.0
Payments         4248.00          4248.00         0.0

Both rows reconcile to a `$0.00` difference — every purchase and payment line on the statement was captured and none were double-counted.

## Export

In [13]:
output_path = Path("Sample2025_transactions_extracted.csv")
validated_df.to_csv(output_path, index=False)
f"Exported {len(validated_df)} transactions to {output_path}"

'Exported 15 transactions to Sample2025_transactions_extracted.csv'